# Tiny Nemotron Pretraining Notebook

This notebook walks through the full pretraining workflow for a tiny Nemotron-style hybrid model:

1. **Setup** — imports & hyperparameters
2. **Tokenizer** — load the Nemotron tokenizer from HuggingFace
3. **Dataset** — stream & pack texts from `HuggingFaceFW/fineweb-edu`
4. **Model** — build a tiny Nemotron (Mamba + Attention + MoE)
5. **Training loop** — JIT-compiled train step with MoE bias updates
6. **Evaluation** — validation loss & perplexity
7. **Checkpointing** — save/load with Orbax
8. **Generation** — autoregressive text generation

## 0. Environment Setup

Run this section first to:
1. Detect whether you are on **Google Colab**, **Kaggle**, or a local machine.
2. Install required packages (JAX, Flax, Optax, Orbax, Datasets, Transformers).
3. Clone the repository so local source modules (`nemotron.py`, `mamba_2.py`, etc.) are importable.

In [ ]:
import os
import sys

# ── Detect runtime environment ─────────────────────────────────────────────────
IN_COLAB  = False
IN_KAGGLE = os.path.exists("/kaggle/working") and os.path.exists("/kaggle/input")

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

ENV_NAME = "Google Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "Local"
print("Running on:", ENV_NAME)

In [ ]:
if IN_COLAB or IN_KAGGLE:
    # ── Accelerator selection ──────────────────────────────────────────────────
    # Set ACCELERATOR to match the hardware you have enabled:
    #   "cuda12" → GPU  (T4 / V100 / A100 on Colab GPU or Kaggle GPU)
    #   "tpu"    → TPU  (Colab TPU runtime only)
    #   "cpu"    → CPU  (no hardware accelerator)
    ACCELERATOR = "cuda"

    if ACCELERATOR == "tpu":
        get_ipython().system('pip install --upgrade "jax[tpu]"')
    elif ACCELERATOR == "cuda":
        get_ipython().system('pip install --upgrade "jax[cuda13]"')
    elif ACCELERATOR == "rocm":
        get_ipython().system('pip install --upgrade "jax[rocm7-local]"')
    else:
        get_ipython().system('pip install --upgrade jax')

    get_ipython().system('pip install --upgrade flax optax orbax-checkpoint datasets transformers matplotlib')
    print("Packages installed.")
else:
    print("Local environment — skipping pip installs.")

In [ ]:
import pathlib

# ── Locate / clone the repository ─────────────────────────────────────────────
REPO_URL = "https://github.com/wisnunugroho21/nugie-jax-nemotron.git"

if IN_COLAB:
    REPO_DIR = pathlib.Path("/content/nugie-jax-nemotron")
    if not REPO_DIR.exists():
        get_ipython().system(f'git clone -b large-reasoning-model {REPO_URL} {REPO_DIR}')
elif IN_KAGGLE:
    REPO_DIR = pathlib.Path("/kaggle/working/nugie-jax-nemotron")
    if not REPO_DIR.exists():
        get_ipython().system(f'git clone -b large-reasoning-model  {REPO_URL} {REPO_DIR}')
else:
    # Local: auto-detect repo root by looking for nemotron.py
    _cwd = pathlib.Path.cwd()
    REPO_DIR = _cwd
    for _candidate in [_cwd, _cwd.parent, _cwd.parent.parent]:
        if (_candidate / "nemotron.py").exists():
            REPO_DIR = _candidate
            break

print("REPO_DIR:", REPO_DIR)

In [ ]:
# ── (Optional) Mount Google Drive for persistent checkpoints on Colab ──────────
# Uncomment the block below if you want checkpoints to survive session restarts.
# Skip this cell on Kaggle or local runs.
#
# if IN_COLAB:
#     from google.colab import drive
#     drive.mount("/content/drive")
#     CHECKPOINT_DIR = "/content/drive/MyDrive/nemotron_checkpoints"
#     print("Checkpoints will be saved to Google Drive:", CHECKPOINT_DIR)

## 1. Imports & Hyperparameters

In [ ]:
import math
import pathlib
import sys

import jax
import jax.numpy as jnp
import numpy as np
import optax
import orbax.checkpoint as ocp
from datasets import load_dataset
from flax import nnx
from transformers import AutoTokenizer

# Make sure the project root is on sys.path so we can import local modules.
# PROJECT_ROOT is set in the '0. Environment Setup' cell above.
PROJECT_ROOT = REPO_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from moe import SparseMoE
from nemotron import NemotronConfig, NemotronNanoBlock

print("JAX devices:", jax.devices())

In [ ]:
# ── Hyperparameters ────────────────────────────────────────────────────────────
# Adjust these to fit your hardware / experiment goals.

VOCAB_SIZE = 131072  # nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16 tokenizer vocabulary
SEQ_LEN = 256  # input tokens per sample — must be divisible by CHUNK_SIZE
CHUNK_SIZE = 64  # Mamba SSD chunk size — must match NemotronConfig.mamba_chunk_size
BATCH_SIZE = 2
CHECKPOINT_EVERY = 200  # save a checkpoint every N training steps
MAX_TRAIN_STEPS = 10000
WARMUP_STEPS = int(MAX_TRAIN_STEPS * 0.1)  # linear warmup for the first N steps
STABLE_STEPS = int(MAX_TRAIN_STEPS * 0.7)
DECAY_STEPS = int(MAX_TRAIN_STEPS * 0.2)
PEAK_LR = 3e-4
MIN_LR = 1e-5
WEIGHT_DECAY = 0.1
B1 = 0.9  # Adam beta1
B2 = 0.95  # Adam beta2
VAL_STEPS = 50  # how many batches to average for validation
if IN_COLAB:
    CHECKPOINT_DIR = "/content/checkpoints"
elif IN_KAGGLE:
    CHECKPOINT_DIR = "/kaggle/working/checkpoints"
else:
    CHECKPOINT_DIR = str(PROJECT_ROOT / "checkpoints")
MAX_GEN_TOKENS   = 200      # max new tokens per generation call
MAX_CTX_LEN      = 512      # rolling context window during generation

assert SEQ_LEN % CHUNK_SIZE == 0,     "SEQ_LEN must be divisible by CHUNK_SIZE"
assert MAX_CTX_LEN % CHUNK_SIZE == 0, "MAX_CTX_LEN must be divisible by CHUNK_SIZE"

print(f"VOCAB_SIZE={VOCAB_SIZE}, SEQ_LEN={SEQ_LEN}, BATCH_SIZE={BATCH_SIZE}")
print(f"MAX_TRAIN_STEPS={MAX_TRAIN_STEPS}, CHECKPOINT_DIR={CHECKPOINT_DIR}")

## 2. Tokenizer

In [ ]:
print("Loading Nemotron tokenizer ...")
tokenizer = AutoTokenizer.from_pretrained(
    "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
)

# This tokenizer has no pad token by default; reuse eos for padding.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Vocabulary size : {tokenizer.vocab_size}")
print(f"EOS token       : {tokenizer.eos_token!r}  (id={tokenizer.eos_token_id})")

## 3. Dataset Helpers

In [ ]:
def load_raw_texts(max_samples: int, skip: int = 0) -> list[str]:
    """Stream texts from HuggingFaceFW/fineweb-edu.

    Uses the 10B-token sample split — no need to download the full dataset.
    `skip` lets training and validation draw from non-overlapping portions.
    """
    print(f"Loading {max_samples} texts (skip={skip}) ...")
    ds = load_dataset(
        "HuggingFaceFW/fineweb-edu",
        split="train",
        streaming=True,
    )
    texts: list[str] = []
    for i, sample in enumerate(ds):
        if i < skip:
            continue
        texts.append(sample["text"])
        if len(texts) >= max_samples:
            break
    print(f"  Got {len(texts)} texts.")
    return texts


def tokenize_and_pack(texts: list[str], tokenizer, seq_len: int) -> np.ndarray:
    """Tokenize all texts, concatenate into one stream, cut into (seq_len+1) chunks.

    Returns shape (n_chunks, seq_len + 1).
    Each chunk:  inputs = chunk[:, :-1], labels = chunk[:, 1:]
    """
    chunk_len = seq_len + 1
    all_tokens: list[int] = []
    for text in texts:
        all_tokens.extend(tokenizer.encode(text))
        all_tokens.append(tokenizer.eos_token_id)
    n = (len(all_tokens) // chunk_len) * chunk_len
    return np.array(all_tokens[:n], dtype=np.int32).reshape(-1, chunk_len)


def make_batches(chunks: np.ndarray, batch_size: int):
    """Shuffle chunks once, then yield (batch_size, chunk_len) arrays."""
    idx = np.random.permutation(len(chunks))
    chunks = chunks[idx]
    for i in range(0, len(chunks) - batch_size + 1, batch_size):
        yield chunks[i : i + batch_size]

In [ ]:
# Stream training texts (2 000) then validation texts (200, non-overlapping).
# Increase max_samples for a longer / better training run.
train_texts = load_raw_texts(max_samples=2000, skip=0)
val_texts   = load_raw_texts(max_samples=200,  skip=2000)

train_chunks = tokenize_and_pack(train_texts, tokenizer, SEQ_LEN)
val_chunks   = tokenize_and_pack(val_texts,   tokenizer, SEQ_LEN)

print(f"Train chunks : {len(train_chunks)}")
print(f"Val   chunks : {len(val_chunks)}")
print(f"Chunk shape  : {train_chunks.shape}  (batch, seq_len+1)")

## 4. Learning Rate Schedule

In [ ]:
def create_lr_schedule(
    peak_lr: float,
    min_lr: float,
    warmup_steps: int,
    stable_steps: int,
    decay_steps: int
) -> optax.Schedule:
    """
    Creates a two-phase learning rate schedule:
        Phase 1 — Warmup  : linear ramp from 0 → peak_lr
        Phase 2 — Stable  : constant at peak_lr
        Phase 3 — Decay   : cosine decay from peak_lr → min_lr

    This avoids early training instability (warmup) while allowing the
    optimizer to fine-tune at smaller learning rates later (cosine decay).
    """
    warmup = optax.linear_schedule(
        init_value=0.0,
        end_value=peak_lr,
        transition_steps=warmup_steps,
    )
    stable = optax.constant_schedule(peak_lr)
    decay = optax.cosine_decay_schedule(
        init_value=peak_lr,
        decay_steps=decay_steps,
        alpha=min_lr / peak_lr,   # final LR = peak_lr * alpha = min_lr
    )

    return optax.join_schedules(
        schedules=[warmup, stable, decay],
        boundaries=[warmup_steps, warmup_steps + stable_steps],
    )

def make_gradient_transform_optimizer(peak_lr: float,
    min_lr: float,
    warmup_steps: int,
    stable_steps: int,
    decay_steps: int, weight_decay: float = 0.1, b1: float = 0.9, b2: float = 0.95) -> optax.GradientTransformation:
    """
    Creates an Optax gradient transformation for optimizer with the custom learning rate schedule.
    We use AdamW with weight decay, which is common for transformer training.
    """
    warmup_steps = max(warmup_steps, 1) # sanity check to avoid division by zero in warmup schedule
    stable_steps = max(stable_steps, 1) # sanity check to avoid division by zero in stable schedule
    decay_steps = max(decay_steps, 1)   # sanity check to avoid division by zero in decay schedule
    
    lr_schedule = create_lr_schedule(
        peak_lr=peak_lr,
        min_lr=min_lr,
        warmup_steps=warmup_steps,
        stable_steps=stable_steps,
        decay_steps=decay_steps,
    )

    return optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.adamw(learning_rate=lr_schedule, weight_decay=weight_decay, b1=b1, b2=b2),
    )

## 5. Model

In [ ]:
def build_model(config: NemotronConfig, seed: int = 0) -> NemotronNanoBlock:
    return NemotronNanoBlock(rngs=nnx.Rngs(seed), config=config)


def collect_moe_layers(model: NemotronNanoBlock) -> list[SparseMoE]:
    """Collect every SparseMoE sub-module in the model."""
    return [block.moe for block in model.blocks]


print("Building model ...")
config = NemotronConfig().from_preset("kaggle")
config.vocab_size = VOCAB_SIZE
config.mamba_chunk_size = CHUNK_SIZE
config.validate()

model      = build_model(config=config, seed=0)
moe_layers = collect_moe_layers(model)

optimizer = nnx.Optimizer(
    model,
    make_gradient_transform_optimizer(
        PEAK_LR,
        MIN_LR,
        WARMUP_STEPS,
        STABLE_STEPS,
        DECAY_STEPS,
        weight_decay=WEIGHT_DECAY,
        b1=B1,
        b2=B2,
    ),
    wrt=nnx.Param,
)

# Count parameters
graphdef, state = nnx.split(model)
n_params = sum(v.size for v in jax.tree_util.tree_leaves(state))
print(f"Parameters : {n_params:,}")

## 6. Loss & Training Step

In [ ]:
def cross_entropy_loss(model: NemotronNanoBlock, batch: jax.Array) -> jax.Array:
    """Standard next-token prediction loss.

    batch: (B, seq_len + 1)
      inputs = batch[:, :-1]  → fed into the model
      labels = batch[:, 1:]   → shifted-by-one targets
    """
    inputs = batch[:, :-1]          # (B, seq_len)
    labels = batch[:, 1:]           # (B, seq_len)
    logits = model(inputs)          # (B, seq_len, vocab_size)
    loss = optax.softmax_cross_entropy_with_integer_labels(logits, labels)
    return loss.mean()


@nnx.jit
def train_step(
    model: NemotronNanoBlock,
    optimizer: nnx.Optimizer,
    batch: jax.Array,
) -> jax.Array:
    """Compute gradients and update model weights. Returns scalar loss."""
    loss, grads = nnx.value_and_grad(
        cross_entropy_loss, argnums=nnx.DiffState(0, nnx.Param)
    )(model, batch)
    optimizer.update(model, grads)
    return loss


def update_moe_biases(moe_layers: list[SparseMoE]) -> None:
    """Update expert load-balancing biases after each optimizer step."""
    for moe in moe_layers:
        moe.update_expert_bias(moe.last_topk_indices.get_value())

## 7. Checkpointing (Orbax)

In [ ]:
def make_checkpoint_manager(
    ckpt_dir: str, max_to_keep: int = 3
) -> ocp.CheckpointManager:
    """Create an Orbax CheckpointManager that retains the last `max_to_keep` steps."""
    options = ocp.CheckpointManagerOptions(max_to_keep=max_to_keep)
    return ocp.CheckpointManager(pathlib.Path(ckpt_dir), options=options)


def save_checkpoint(
    manager: ocp.CheckpointManager,
    model: NemotronNanoBlock,
    step: int,
) -> None:
    """Save model state at `step`."""
    _, state = nnx.split(model)
    manager.save(step, args=ocp.args.StandardSave(state))
    manager.wait_until_finished()
    print(f"  Checkpoint saved: step {step}")


def load_latest_checkpoint(
    manager: ocp.CheckpointManager,
    model: NemotronNanoBlock,
    config: NemotronConfig,
) -> int:
    """Restore the most recent checkpoint. Returns the step number (0 if none)."""
    latest = manager.latest_step()
    if latest is None:
        return 0
    abstract_model = nnx.eval_shape(
        lambda: NemotronNanoBlock(rngs=nnx.Rngs(0), config=config)
    )
    _, abs_state = nnx.split(abstract_model)
    restored = manager.restore(latest, args=ocp.args.StandardRestore(abs_state))
    nnx.update(model, restored)
    print(f"  Resumed from checkpoint at step {latest}")
    return latest


# Create manager and resume from latest checkpoint (if any).
ckpt_manager = make_checkpoint_manager(CHECKPOINT_DIR)
start_step   = load_latest_checkpoint(ckpt_manager, model, config)
print(f"Starting from step {start_step}")

## 8. Training Loop

In [ ]:
print(
    f"Training for {MAX_TRAIN_STEPS} steps "
    f"(batch={BATCH_SIZE}, seq_len={SEQ_LEN}) ..."
)
print("(First step is slow — JAX JIT-compiles the model.)\n")

step       = start_step
batch_iter = iter(make_batches(train_chunks, BATCH_SIZE))
loss_history: list[float] = []

while step < MAX_TRAIN_STEPS:
    try:
        batch_np = next(batch_iter)
    except StopIteration:
        batch_iter = iter(make_batches(train_chunks, BATCH_SIZE))
        batch_np   = next(batch_iter)

    loss = train_step(model, optimizer, jnp.array(batch_np))
    update_moe_biases(moe_layers)

    step += 1
    loss_val = float(loss)
    loss_history.append(loss_val)

    if step % 10 == 0:
        print(f"  step {step:5d} / {MAX_TRAIN_STEPS}  |  loss {loss_val:.4f}")

    if step % CHECKPOINT_EVERY == 0:
        save_checkpoint(ckpt_manager, model, step)

print("\nTraining complete.")

In [ ]:
# Plot the training loss curve.
try:
    import matplotlib.pyplot as plt

    plt.figure(figsize=(10, 4))
    plt.plot(range(start_step + 1, start_step + len(loss_history) + 1), loss_history)
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.title("Training Loss")
    plt.grid(True)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed — skipping loss plot.")

## 9. Evaluation

In [ ]:
def evaluate(
    model: NemotronNanoBlock,
    val_chunks: np.ndarray,
    val_steps: int,
) -> tuple[float, float]:
    """Return (mean_loss, perplexity) averaged over val_steps batches."""
    total_loss = 0.0
    count = 0
    for batch_np in make_batches(val_chunks, BATCH_SIZE):
        if count >= val_steps:
            break
        loss = cross_entropy_loss(model, jnp.array(batch_np))
        total_loss += float(loss)
        count += 1
    mean_loss  = total_loss / max(count, 1)
    perplexity = math.exp(min(mean_loss, 20))  # clamp to avoid overflow on a fresh model
    return mean_loss, perplexity


print("Evaluating on validation set ...")
val_loss, val_ppl = evaluate(model, val_chunks, VAL_STEPS)
print(f"  Validation loss : {val_loss:.4f}")
print(f"  Perplexity      : {val_ppl:.2f}")

## 10. Save Final Checkpoint

In [ ]:
save_checkpoint(ckpt_manager, model, step)
ckpt_manager.close()
print("Done.")

## 11. Text Generation

In [ ]:
def generate(
    model: NemotronNanoBlock,
    tokenizer,
    prompt: str,
    max_new_tokens: int = MAX_GEN_TOKENS,
    temperature: float = 0.8,
    rng_seed: int = 42,
) -> str:
    """Generate text autoregressively with temperature sampling.

    The context is left-padded to the nearest multiple of CHUNK_SIZE before
    each forward pass (Mamba SSD requires seqlen % chunk_size == 0).
    """
    prompt_tokens = tokenizer.encode(prompt)
    tokens = list(prompt_tokens)
    rng = jax.random.PRNGKey(rng_seed)

    for _ in range(max_new_tokens):
        ctx = tokens[-MAX_CTX_LEN:]

        # Left-pad so length is a multiple of CHUNK_SIZE.
        pad_len = (-len(ctx)) % CHUNK_SIZE
        padded  = [tokenizer.eos_token_id] * pad_len + ctx
        if not padded:
            padded = [tokenizer.eos_token_id] * CHUNK_SIZE

        input_ids  = jnp.array([padded])       # (1, padded_len)
        logits     = model(input_ids)          # (1, padded_len, vocab_size)
        next_logits = logits[0, -1, :]         # (vocab_size,)

        rng, sample_rng = jax.random.split(rng)
        next_token = int(
            jax.random.categorical(sample_rng, next_logits / temperature)
        )
        tokens.append(next_token)

        if next_token == tokenizer.eos_token_id:
            break

    return tokenizer.decode(tokens[len(prompt_tokens):])

In [ ]:
# ── Try a quick generation ─────────────────────────────────────────────────────
# Note: after only 1 000 steps on a tiny model the output will be incoherent.
# Increase MAX_TRAIN_STEPS and train on more data for meaningful results.

PROMPT = "The quick brown fox"

print(f"Prompt : {PROMPT!r}\n")
response = generate(
    model,
    tokenizer,
    prompt=PROMPT,
    max_new_tokens=80,
    temperature=0.8,
    rng_seed=0,
)
print(f"Output : {response}")

In [ ]:
# ── Try multiple prompts / temperatures ───────────────────────────────────────
prompts      = ["The capital of France is", "The meaning of life is", "Once upon a time"]
temperatures = [0.5, 0.8, 1.0]

for i, prompt in enumerate(prompts):
    temp = temperatures[i % len(temperatures)]
    out  = generate(model, tokenizer, prompt, max_new_tokens=50, temperature=temp, rng_seed=i)
    print(f"[T={temp}] {prompt!r}\n  → {out}\n")